In [ ]:
from langgraph.graph import START, END, StateGraph, MessagesState
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from typing import Literal

In [ ]:
llm = ChatOpenAI(
    base_url="http://localhost:1234/api/v1",
    api_key="not-needed",
    model="google/gemma-3-4b",
    temperature=0,
    max_completion_tokens=250
)

In [ ]:
class State(MessagesState):
    summary: str
# Not annotated with a reducer, with each update, the old value of the summary key will replaced with a new one

define the nodes


In [ ]:
# ask_question node
def ask_question(state: State) -> State:

    print(f"\n------->  ENTERING ask_question:")

    question = "what is your question?"
    print(question)

    return State(messages= [AIMessage(question), HumanMessage(input())])

In [ ]:
#chatbot node
def chatbot(state: State) -> State:  #this function is good when the number of nodes increasing
    
    print(f"\n------->  ENTERING chatbot:")
    for i in state["messages"]:
        i.pretty_print()

    system_message = f'''
    Here is a quick summary of what's been discussed so far:
    {state.get("summary", "")}
    
    keep this in mind as you answer the next question
    '''

    response = llm.invoke([SystemMessage(system_message)] + state["messages"])
    response.pretty_print()

    return State(messages= [response])

In [ ]:
# ask_another_question node 
def ask_another_question(state: State) -> State:

    print(f"\n------->  ENTERING ask_another_question:")

    question = "would you like to ask another question (yes/no)?"
    print(question)

    return State(messages= [AIMessage(question), HumanMessage(input())])

In [ ]:
# summarize node
def summarize_and_delete_messages(state: State) -> State:

    print(f"\n------->  ENTERING summarized_messages:")

    new_conversations = ""
    for i in state["messages"]:
        new_conversations += f"{i.type}: {i.content}\n\n"

    summary_instructions = f'''
    update the ongoing summary by incorporating the new lines of conversation below.
    build upon the pervious summary rather than repeating it so that the result reflects the most recent context
    and developments

    pervious summary:
    {state.get("summary", "")}

    new conversation:
    {new_conversations}
    '''
    print(summary_instructions)

    summary = llm.invoke([HumanMessage(summary_instructions)])

    remove_messages = [RemoveMessage(id= i.id) for i in state["messages"][:]]

    return State(messages= remove_messages, summary= summary.content)

In [ ]:
# Define routing function, conditional edges
def routing_function(state: State) -> Literal["summarize_and_delete_messages", "__end__"]: #or can put str instead of Literal[...]

    if state["messages"][-1].content == "yes": #check last message to append new message, change from 0 to -1
        return "summarize_and_delete_messages"
    else:
        return "__end__"

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_node("summarize_and_delete_messages", summarize_and_delete_messages)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_edge("summarize_and_delete_messages", "ask_question")
graph.add_conditional_edges(source= "ask_another_question", 
                            path= routing_function)

In [ ]:
graph_compiled = graph.compile() 

In [ ]:
graph_compiled.invoke(State(messages=[]))